In [78]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [79]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

## Deep dive into our dataset.

When analysing the dataset from `FibonacciModDataset` in our `model_dim_4_layer_3.py` we noticed that loss for evulate set is less than the loss for training set, which is quite the opposite.

In [80]:
from script.model_dim_4_layer_3 import FibonacciModDataset
from torch.utils.data import Dataset, DataLoader, random_split


seq_len = 20
batch_size = 6
vocab_size=10
generated_ds = FibonacciModDataset(num_samples=25000, mod=vocab_size, seq_len=seq_len)

train_size = int(0.8 * len(generated_ds))
test_size = len(generated_ds) - train_size
train_ds, test_ds = random_split(generated_ds, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)

c = 0

for x, y in train_ds:
    if c >= 5:
        break
    print(x, y)

    c += 1



tensor([1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8]) tensor([4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8, 1])
tensor([5, 4, 9, 3, 2, 5, 7, 2, 9, 1, 0, 1, 1, 2, 3, 5, 8, 3, 1, 4]) tensor([4, 9, 3, 2, 5, 7, 2, 9, 1, 0, 1, 1, 2, 3, 5, 8, 3, 1, 4, 5])
tensor([0, 9, 9, 8, 7, 5, 2, 7, 9, 6, 5, 1, 6, 7, 3, 0, 3, 3, 6, 9]) tensor([9, 9, 8, 7, 5, 2, 7, 9, 6, 5, 1, 6, 7, 3, 0, 3, 3, 6, 9, 5])
tensor([7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8, 1, 9, 0, 9, 9, 8, 7, 5]) tensor([7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8, 1, 9, 0, 9, 9, 8, 7, 5, 2])
tensor([3, 1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3]) tensor([1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8])



Basically we are generating random shamples and then using sliding window to generate shamples from those.
Now the randomness is in the start_idx, we want to see the dublicates pair to better understand it.
So we want to see the dublicates in both train and test set pais because, if the model has already seen
the data then it will try to find a shortcut

In [81]:
import torch

print(len(train_ds))
print(len(test_ds))
print(len(train_ds) + len(test_ds)) # that should correspond to the shamples provided.

pairs = []
for x, y in train_ds:
    x = torch.cat((x, y[-1: ]))
    pairs.append(x)
print(pairs[0: 4])
print(len(pairs)) # just making sure the math match
print(len(pairs[0]))


20000
5000
25000
[tensor([1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8, 1]), tensor([5, 4, 9, 3, 2, 5, 7, 2, 9, 1, 0, 1, 1, 2, 3, 5, 8, 3, 1, 4, 5]), tensor([0, 9, 9, 8, 7, 5, 2, 7, 9, 6, 5, 1, 6, 7, 3, 0, 3, 3, 6, 9, 5]), tensor([7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8, 1, 9, 0, 9, 9, 8, 7, 5, 2])]
20000
21


## Algorithmic thinking for fun!

We need to find the dublicate pairs, now so what I want to do is, it's not just about dublicate pairs that we are finding but we are finding pairs in a fibonachi sequence. 

In [102]:
arr = torch.cat(pairs, dim=0).tolist()
arr = arr[0: 21]
print(arr)

pair_map = []
for i in range(0, len(arr), 3):
    ''' 
        Here since its a fibonachi sequence so a + b = c characterstics
        so its not a common pair problem.
    '''
    a = arr[i]
    b = arr[i + 1]

    see_pairs = next((item for item in pair_map if item.get('a') + item.get('b') == a+ b ), None)

    if see_pairs is None: # if not exists then append there.
        pair_map.append({
            'a': a,
            'b': b,
            'f': 1
        })
    c1 = a + b
    for j in range(i, len(arr), 3):
        try:
            ca = arr[j + 3] # because we skip that output part.
            cb = arr[j + 4]
            c2 = ca + cb

            if c1 == c2:
                # if found pairs then we are going to increase the frequency.
                ''' If we are looking for dub of this see_pairs particular pairs then
                 if course we can increment that see_pairs and go on '''
                see_pairs['f'] += 1
        except Exception: # we might go off limit for sure.
            pass
result = []
for i in range(0, len(pair_map)):
    if pair_map[i].get('f') > 1:
        result.append(pair_map[i])

print(result)


[1, 4, 5, 9, 4, 3, 7, 0, 7, 7, 4, 1, 5, 6, 1, 7, 8, 5, 3, 8, 1]
[{'a': 7, 'b': 4, 'f': 2}]
